In [1]:
from allensdk.core.cell_types_cache import CellTypesCache
from allensdk.ephys.extract_cell_features import extract_cell_features
from collections import defaultdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os
import pandas as pd
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from allensdk.core.cell_types_cache import CellTypesCache

path_name = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/'

df = pd.read_csv(os.path.join(path_name,'data','cell_types_specimen_details.csv'))


In [2]:
ctc = CellTypesCache(manifest_file='cell_types/manifest.json')


In [3]:
def exp_func(t, tau, A, C):
    return A * (1 - np.exp(-t / tau)) + C

In [4]:
def extract_ephys(specimen_ids): #, ax):

    ephys_features = dict()

    for i, specimen_id in enumerate(specimen_ids.values):

        # download the ephys data and sweep metadata
        data_set = ctc.get_ephys_data(specimen_id)
        sweeps = ctc.get_ephys_sweeps(specimen_id)

        # group the sweeps by stimulus 
        sweep_numbers = defaultdict(list)
        for sweep in sweeps:
            sweep_numbers[sweep['stimulus_name']].append(sweep['sweep_number'])

        # calculate features

        ephys_features[specimen_id] = dict()
        ephys_features[specimen_id]['index'] = specimen_id

        try: 
            cell_features = extract_cell_features(data_set,
                                                sweep_numbers['Ramp'],
                                                sweep_numbers['Short Square'],
                                                sweep_numbers['Long Square'])
            # calculate features
            long_square = cell_features['long_squares']

            # Assuming your dictionary is named 'data'
            long_df = pd.DataFrame(long_square['sweeps'])

            # mebrane potential
            ephys_features[specimen_id]['RMP'] = np.mean(long_df.v_baseline.values)

            # sag
            ephys_features[specimen_id]['sag'] = np.mean(long_df['sag'][long_df['stim_amp'] < 0].values)

            

            # get tau amps
            ephys_features[specimen_id]['sag_stim_amp'] = long_df['stim_amp'][long_df['stim_amp'] < 0].values/1000

            # ISIs positive amplitude
            ephys_features[specimen_id]['max_amp_isi'] = np.max(long_df['stim_amp'])/1000

            # Action potential features
            short_square = cell_features['short_squares']
            ephys_features[specimen_id]['upstroke_downstroke'] = short_square['common_amp_sweeps'][0]['spikes'][0]['upstroke_downstroke_ratio']
            ephys_features[specimen_id]['ap_width'] = short_square['common_amp_sweeps'][0]['spikes'][0]['width']*1000
            ephys_features[specimen_id]['ap_threshold'] = short_square['common_amp_sweeps'][0]['spikes'][0]['threshold_v']

            sweeps = data_set.get_experiment_sweep_numbers()
            sweeps_meta = dict()
            for sweep in sweeps:
                sweeps_meta[sweep] = data_set.get_sweep_metadata(sweep)

            sweeps_df = pd.DataFrame.from_dict(sweeps_meta, orient='index')

            # get voltage trace
            sweeps_long = sweeps_df[(sweeps_df['aibs_stimulus_name'] == b'Long Square') & (sweeps_df['aibs_stimulus_amplitude_pa'] >0)]
            num_spks = np.zeros_like(sweeps_long.index)
            ampl = np.zeros_like(sweeps_long.index)
            first_isi = np.zeros_like(sweeps_long.index)
            mean_isi = np.zeros_like(sweeps_long.index)
            for i,s in enumerate(sweeps_long.index):
                sweep = data_set.get_sweep(s)

                # get firing rate slope
                v = sweep['response']
                pk = find_peaks(v,height=0)
                num_spks[i] = len(pk[0])
                ampl[i] = sweeps_long['aibs_stimulus_amplitude_pa'][s]

                dt = 1.0e3 / sweep['sampling_rate']
                num_samples = len(v)
                t = np.arange(num_samples) * dt

                # find ISI because the units from Allen seem off
                if len(pk[0]) > 1:
                    num_spks[i] = len(pk[0])
                    ampl[i] = sweeps_long['aibs_stimulus_amplitude_pa'][s]

                    dt = 1.0e3 / sweep['sampling_rate']
                    num_samples = len(v)
                    t = np.arange(num_samples) * dt

                    first_isi[i] = np.diff(t[pk[0]])[0]
                    mean_isi[i] = np.mean(np.diff(t[pk[0]]))
            
            # firing rate slope
            ephys_features[specimen_id]['FR_slope'] = np.polyfit(ampl[num_spks >0],num_spks[num_spks >0],1)[0]
            ephys_features[specimen_id]['first_isi'] = np.median(first_isi[first_isi>0])
            ephys_features[specimen_id]['mean_isi'] = np.median(first_isi[mean_isi>0])

            # mebrane time constant based on hyperpolarization experiments
            sweeps_long = sweeps_df[(sweeps_df['aibs_stimulus_name'] == b'Long Square') & (sweeps_df['aibs_stimulus_amplitude_pa'] < 0)]
            tau_ms = np.zeros_like(sweeps_long.index)
            for i,s in enumerate(sweeps_long.index):
                sweep = data_set.get_sweep(s)
                
                # get membrane time constant
                t_max = len(sweep['response'])/sweep['sampling_rate']*1000          # convert seconds to ms
                timevec = np.linspace(0, t_max, len(sweep['response']))

                stim_onset = np.where(sweep['stimulus']<0)[0][0]
                stim_offset = np.where(sweep['stimulus'][stim_onset:]>=0)[0][0]+stim_onset

                t_rise = timevec[stim_onset:stim_offset]-timevec[stim_onset]
                v_rise = sweep['response'][stim_onset:stim_offset]
                # baseline correction
                delta_v = np.mean(v_rise) - np.mean(sweep['response'][:stim_onset])
                popt, pcov = curve_fit(exp_func, t_rise, v_rise, p0=[10, abs(delta_v), v_rise[0]])
                tau_ms[i] = popt[0]
                        
            # tau
            ephys_features[specimen_id]['tau'] = np.mean(tau_ms)
            
            del sweep, v, ampl, short_square, long_square, long_df, data_set

        except:

            ephys_features[specimen_id]['RMP'] = float('nan')
            ephys_features[specimen_id]['sag'] = float('nan')
            ephys_features[specimen_id]['mean_isi'] = float('nan')
            ephys_features[specimen_id]['first_isi'] = float('nan')
            ephys_features[specimen_id]['upstroke_downstroke'] = float('nan')
            ephys_features[specimen_id]['ap_width'] = float('nan')
            ephys_features[specimen_id]['ap_threshold'] = float('nan')

    return ephys_features


In [5]:
## Code to develop membrane time constant extraction

# # extract time membrane constant the same way that I'm doing it in the model

# # download the ephys data and sweep metadata
# specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '2')]
# specimen_id = specimen_ids.iloc[0]

# data_set = ctc.get_ephys_data(specimen_id)


# sweeps = data_set.get_experiment_sweep_numbers()
# sweeps_meta = dict()
# for sweep in sweeps:
#     sweeps_meta[sweep] = data_set.get_sweep_metadata(sweep)

# sweeps_df = pd.DataFrame.from_dict(sweeps_meta, orient='index')

# # get voltage trace
# sweeps_long = sweeps_df[(sweeps_df['aibs_stimulus_name'] == b'Long Square') & (sweeps_df['aibs_stimulus_amplitude_pa'] <0)]

# s= sweeps_long.index[0]
# sweep = data_set.get_sweep(s)


# t_max = len(sweep['response'])/sweep['sampling_rate']*1000
# timevec = np.linspace(0, t_max, len(sweep['response']))

# stim_onset = np.where(sweep['stimulus']<0)[0][0]
# stim_offset = np.where(sweep['stimulus'][stim_onset:]>=0)[0][0]+stim_onset

# t_rise = timevec[stim_onset:stim_offset]-timevec[stim_onset]
# v_rise = sweep['response'][stim_onset:stim_offset]
# # baseline correction
# delta_v = np.mean(v_rise) - np.mean(sweep['response'][:stim_onset])
# popt, pcov = curve_fit(exp_func, t_rise, v_rise, p0=[10, abs(delta_v), v_rise[0]])
# tau_ms = popt[0]


# fig, ax = plt.subplots(2,1, figsize=(10, 6))
# ax[0].plot(timevec, sweep['stimulus'])
# ax[0].plot(timevec[stim_onset], sweep['stimulus'][stim_onset], 'ro')
# ax[0].plot(timevec[stim_offset], sweep['stimulus'][stim_offset], 'go')
# ax[1].plot(timevec, sweep['response'])

In [6]:
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '2')]
specimen_id = specimen_ids.iloc[0]

# download the ephys data and sweep metadata
data_set = ctc.get_ephys_data(specimen_id)
sweeps = ctc.get_ephys_sweeps(specimen_id)

# group the sweeps by stimulus 
sweep_numbers = defaultdict(list)
for sweep in sweeps:
    sweep_numbers[sweep['stimulus_name']].append(sweep['sweep_number'])

# calculate features

cell_features = extract_cell_features(data_set,
                                    sweep_numbers['Ramp'],
                                    sweep_numbers['Short Square'],
                                    sweep_numbers['Long Square'])
# calculate features
long_square = cell_features['long_squares']

# Assuming your dictionary is named 'data'
long_df = pd.DataFrame(long_square['sweeps'])

# Set 'id' as the index
long_df.set_index('id', inplace=True)

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]


In [7]:

# inhibitory neurons first
os.chdir(os.path.join(path_name,'data','interneuron'))

# inhibitory
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'aspiny')]
ephys_features = extract_ephys(specimen_ids)
ephys_inh = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_inh.to_csv(os.path.join(path_name,'data','interneuron','ephys_inh.csv'))
ephys_inh.to_pickle(os.path.join(path_name,'data','interneuron','ephys_inh.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean o

In [8]:

# L2 cells
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '2')]
ephys_features = extract_ephys(specimen_ids)

ephys_l2 = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_l2.to_csv(os.path.join(path_name,'data','l23','ephys_l2.csv'))
ephys_l2.to_pickle(os.path.join(path_name,'data','l23','ephys_l2.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning:

In [9]:
# L3 cells
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '3')]

ephys_features = extract_ephys(specimen_ids)

ephys_l3 = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_l3.to_csv(os.path.join(path_name,'data','l23','ephys_l3.csv'))
ephys_l3.to_pickle(os.path.join(path_name,'data','l23','ephys_l3.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning:

In [10]:
# L5 cells
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '5')]

ephys_features = extract_ephys(specimen_ids)

ephys_l5 = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_l5.to_csv(os.path.join(path_name,'data','l5','ephys_l5.csv'))
ephys_l5.to_pickle(os.path.join(path_name,'data','l5','ephys_l5.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_features.py:820: RuntimeWarning: 